# Count Sentence Dataset Examples

Counts examples for BS/Gridworld sentence datasets split into `deceptive` and `truthful`.

In [ ]:
from collections import Counter
from pathlib import Path
import json

CANDIDATE_ROOTS = [
    Path('/work/users/s/m/smerrill/deception2'),
    Path('/playpen-ssd/smerrill/deception2'),
]
ROOT = next((p for p in CANDIDATE_ROOTS if p.exists()), CANDIDATE_ROOTS[0])
print(f'Using ROOT: {ROOT}')

DATASETS = [
    {
        'game': 'BS',
        'model': 'DeepSeek-R1-Distill-Qwen-7B',
        'split': 'deceptive',
        'path': ROOT / 'BS/Results/SentencePipeline/v1/DeepSeek-R1-Distill-Qwen-7B_deceptive/examples.jsonl',
    },
    {
        'game': 'BS',
        'model': 'DeepSeek-R1-Distill-Qwen-7B',
        'split': 'truthful',
        'path': ROOT / 'BS/Results/SentencePipeline/v1/DeepSeek-R1-Distill-Qwen-7B_truthful/examples.jsonl',
    },
    {
        'game': 'BS',
        'model': 'DeepSeek-R1-Distill-Qwen-14B',
        'split': 'deceptive',
        'path': ROOT / 'BS/Results/SentencePipeline/v1/DeepSeek-R1-Distill-Qwen-14B_deceptive/examples.jsonl',
    },
    {
        'game': 'BS',
        'model': 'DeepSeek-R1-Distill-Qwen-14B',
        'split': 'truthful',
        'path': ROOT / 'BS/Results/SentencePipeline/v1/DeepSeek-R1-Distill-Qwen-14B_truthful/examples.jsonl',
    },
    {
        'game': 'Gridworld',
        'model': 'DeepSeek-R1-Distill-Qwen-7B',
        'split': 'deceptive',
        'path': ROOT / 'Gridworld/Results/SentencePipeline/v1/deepseek-ai_DeepSeek-R1-Distill-Qwen-7B_deceptive/examples.jsonl',
    },
    {
        'game': 'Gridworld',
        'model': 'DeepSeek-R1-Distill-Qwen-7B',
        'split': 'truthful',
        'path': ROOT / 'Gridworld/Results/SentencePipeline/v1/deepseek-ai_DeepSeek-R1-Distill-Qwen-7B_truthful/examples.jsonl',
    },
    {
        'game': 'Gridworld',
        'model': 'DeepSeek-R1-Distill-Qwen-14B',
        'split': 'deceptive',
        'path': ROOT / 'Gridworld/Results/SentencePipeline/v1/deepseek-ai_DeepSeek-R1-Distill-Qwen-14B_deceptive/examples.jsonl',
    },
    {
        'game': 'Gridworld',
        'model': 'DeepSeek-R1-Distill-Qwen-14B',
        'split': 'truthful',
        'path': ROOT / 'Gridworld/Results/SentencePipeline/v1/deepseek-ai_DeepSeek-R1-Distill-Qwen-14B_truthful/examples.jsonl',
    },
]


In [ ]:
def count_examples(path: Path):
    n_examples = 0
    deceptive_counter = Counter()

    if not path.exists():
        return n_examples, deceptive_counter

    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            n_examples += 1
            rec = json.loads(line)
            deceptive_counter[rec.get('deceptive', 'missing')] += 1

    return n_examples, deceptive_counter


rows = []
for ds in DATASETS:
    n_examples, counter = count_examples(ds['path'])
    rows.append({
        'game': ds['game'],
        'model': ds['model'],
        'split': ds['split'],
        'exists': ds['path'].exists(),
        'n_examples': n_examples,
        'deceptive_true': counter.get(True, 0),
        'deceptive_false': counter.get(False, 0),
        'missing_label': counter.get('missing', 0),
        'path': str(ds['path']),
    })

headers = ['game', 'model', 'split', 'exists', 'n_examples', 'deceptive_true', 'deceptive_false', 'missing_label']
widths = {h: max(len(h), max((len(str(r[h])) for r in rows), default=0)) for h in headers}

def fmt_row(r):
    return ' | '.join(str(r[h]).ljust(widths[h]) for h in headers)

print(fmt_row({h: h for h in headers}))
print('-+-'.join('-' * widths[h] for h in headers))
for r in rows:
    print(fmt_row(r))

missing = [r for r in rows if not r['exists']]
if missing:
    print()
    print('Missing datasets:')
    for r in missing:
        print(f"- {r['game']} {r['model']} {r['split']}: {r['path']}")
else:
    print()
    print('All expected datasets are present.')
